# Data Description
- Author: RUEI-HSIANG, CHANG（張睿庠，B12303030；AI協作）
- Date first created: 20261018
- Background: 三類公職名冊與社群存取稽核分表保存，供後續選樣及查證。
- Unit of observation: 各檔分別為議員登記、來源名錄列示議員、第11屆立委、候選人×平台、一次存取。
- Sample period: 2026登記及2026-10-18名錄快照；社群完整窗口尚未取得。
- Known issues: CEC登記不代表最終資格；MOI任期空白；立院10位離職另列；未核實帳號與未取得貼文不能當作0。未把名錄與候選人自動姓名配對，避免同名異人。
- Definition for each variable: 下方 dictionaries 逐檔逐欄列明；空字串代表未取得／未核實或來源空白，依欄位說明解讀。
- Data path: 下方 INPUT_HASHES 五個精確CSV路徑。
- Source data: `data/raw/online_20261018/`各檔與既有 `candidate_campaign_sources.csv`、`candidates_2026_registration.csv`，精確清單與hash見產製Notebook。
- Processing code: [資料整理](../code/prepare_council_and_meta_20261018.ipynb)。
- Role: preparation/source-directory/worklist/access-audit；不是最終分析資料。
- Verification status: 乾淨kernel執行，重讀、欄位、筆數、唯一鍵、22縣市、現任／離職分離均通過。
- SHA-256: 下方 INPUT_HASHES 逐檔固定。
- Downstream analysis input: 同一份明列CSV；須先確認樣本與編碼，尚未做群間比較或模型估計。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


## Change for Your Own Data
逐檔明訂資料路徑與hash；不使用最新檔案或萬用字元替代版本。

In [2]:
INPUT_HASHES = {'data/processed/council_candidates_2026_registration.csv': 'e24bba8e93f1f2e58e4d0d180fc74a9cf615930753070dd71cd3e1312ece829b', 'data/processed/current_councilors_20261018.csv': '83ef5ee391fbba602f04ecf316f2aed5cc8f0eb259b5d222e5f1c6b1593d6c67', 'data/processed/legislators_term11_directory_20261018.csv': 'c9c414a95af141dfd95819d8ef672ee6c09a7d88284373d6e179ff54b65d5b5a', 'data/processed/meta_candidate_account_worklist_20261018.csv': 'ffdce67367497fbe3776d907fe66a71dde6017cb68832b56df9ce09c6306c9fd', 'data/processed/meta_access_attempts_20261018.csv': 'e265c4cf7e81381b2c56ede0029e5b9228aa245f1fa0320edc97cd11ba72beaf'}
data={}
for rel,expected in INPUT_HASHES.items():
    p=project_file(rel,'processed')
    assert sha256_file(p)==expected,rel
    data[Path(rel).stem]=pd.read_csv(p,dtype=str,keep_default_na=False)


## Summary


In [3]:
display(pd.DataFrame([{'file':k,'N':len(v),'columns':len(v.columns)} for k,v in data.items()]))
DEFINITIONS = {'candidate_key': '2026|公職|選區|移除排版空白姓名；僅識別登記，非終生人ID', 'election_year': '登記選舉年', 'office_type': '來源公職類別', 'county_city': '官方縣市名', 'electoral_district': '官方選區原文', 'registration_date_roc': '民國登記日期，保留原文', 'candidate_name': '僅移除排版空白的姓名；不作模糊配對', 'candidate_name_original': 'PDF姓名儲存格文字；排版換行改為空格，逐字原版保留於raw表格JSON', 'party_recommendation_original': 'PDF推薦政黨文字；排版換行改為空格，不推定黨籍', 'party_recommendation_joined': '另欄移除政黨名稱中的排版空白', 'source_note': '來源備註；空白代表原欄空白', 'source_file': '專案相對來源檔案', 'source_page': '原PDF頁碼（1起算）', 'source_url': '官方PDF下載網址', 'qualification_status': 'registration_only_not_final_qualification：未逐人核實最終資格', 'captured_date': '台灣日期2026-10-18，擷取日非原件發布日', 'official_key': '內政部個人頁_PARENT_ID', 'official_name': '內政部名錄姓名', 'organization_and_position': '名錄機關及職稱原文', 'party_original': '名錄黨籍原文，無保留為無', 'profile_url': '官方個人頁，可續查學經歷', 'status': '名錄列示狀態，非逐人任免查證', 'term_start': '未取得，空白，不推定', 'term_end': '未取得，空白，不推定', 'term_note': '任期缺漏說明', 'legislator_key': '立院個人頁nodeid', 'name': '立院名錄姓名', 'term': '第11屆', 'membership_status': 'current現任區／former離職區', 'party_logo_url': '原頁黨徽連結；不從圖片推定黨籍', 'platform': 'Facebook／Instagram／Threads', 'official_urls': '既有官方直接連出帳號；多個以分號保存', 'discovered_on': '官方帳號連結出處網址', 'account_status': '已存官方連結／未核實；未核實不等於沒有帳號', 'posts_status': 'not_collected_not_zero；未取得貼文', 'window_count': '完整窗口貼文量未取得，空白', 'followers_start': '窗口起點追蹤數未取得，空白', 'followers_end': '窗口終點追蹤數未取得，空白', 'updated_date': '工作單更新日期', 'subject': '存取對象或官方工具名稱', 'url': '本次實際測試網址', 'http_status': 'HTTP回應碼；200不代表取得貼文', 'return_code': 'curl退出碼', 'error': 'curl錯誤文字；成功時空白', 'interpretation': '僅回應快照，不是經驗證貼文', 'posts_collected': '未知留空，不以0代表無發文', 'complete_window': 'False，尚未蒐集完整窗口', 'path': '回應快照專案相對路徑', 'sha256': '回應快照SHA-256', 'response_title': '解析HTML title', 'visible_text_chars': '移除標籤後可解析文字字數；不是貼文長度', 'content_status': '官方說明／頁框／存取挑戰，均非完整窗口貼文'}
dictionaries={}
for name,df in data.items():
    assert all(c in DEFINITIONS for c in df.columns),name
    dictionary=pd.DataFrame({'variable':df.columns,'definition':[DEFINITIONS[c] for c in df.columns]})
    dictionaries[name]=dictionary
    print(name);display(dictionary)


,file,N,columns
0,council_candidates_2026_registration,1502,16
1,current_councilors_20261018,872,13
2,legislators_term11_directory_20261018,123,8
3,meta_candidate_account_worklist_20261018,243,11
4,meta_access_attempts_20261018,13,14


council_candidates_2026_registration


,variable,definition
0,candidate_key,2026|公職|選區|移除排版空白姓名；僅識別登記，非終生人ID
1,election_year,登記選舉年
2,office_type,來源公職類別
3,county_city,官方縣市名
4,electoral_district,官方選區原文
5,registration_date_roc,民國登記日期，保留原文
6,candidate_name,僅移除排版空白的姓名；不作模糊配對
7,candidate_name_original,PDF姓名儲存格文字；排版換行改為空格，逐字原版保留於raw表格JSON
8,party_recommendation_original,PDF推薦政黨文字；排版換行改為空格，不推定黨籍
9,party_recommendation_joined,另欄移除政黨名稱中的排版空白


current_councilors_20261018


,variable,definition
0,official_key,內政部個人頁_PARENT_ID
1,official_name,內政部名錄姓名
2,county_city,官方縣市名
3,organization_and_position,名錄機關及職稱原文
4,party_original,名錄黨籍原文，無保留為無
5,office_type,來源公職類別
6,profile_url,官方個人頁，可續查學經歷
7,source_file,專案相對來源檔案
8,captured_date,台灣日期2026-10-18，擷取日非原件發布日
9,status,名錄列示狀態，非逐人任免查證


...

,variable,definition
0,legislator_key,立院個人頁nodeid
1,name,立院名錄姓名
2,term,第11屆
3,membership_status,current現任區／former離職區
4,party_logo_url,原頁黨徽連結；不從圖片推定黨籍
5,profile_url,官方個人頁，可續查學經歷
6,source_file,專案相對來源檔案
7,captured_date,台灣日期2026-10-18，擷取日非原件發布日


meta_candidate_account_worklist_20261018


,variable,definition
0,candidate_key,2026|公職|選區|移除排版空白姓名；僅識別登記，非終生人ID
1,candidate_name,僅移除排版空白的姓名；不作模糊配對
2,platform,Facebook／Instagram／Threads
3,official_urls,既有官方直接連出帳號；多個以分號保存
4,discovered_on,官方帳號連結出處網址
5,account_status,已存官方連結／未核實；未核實不等於沒有帳號
6,posts_status,not_collected_not_zero；未取得貼文
7,window_count,完整窗口貼文量未取得，空白
8,followers_start,窗口起點追蹤數未取得，空白
9,followers_end,窗口終點追蹤數未取得，空白


meta_access_attempts_20261018


,variable,definition
0,subject,存取對象或官方工具名稱
1,url,本次實際測試網址
2,captured_date,台灣日期2026-10-18，擷取日非原件發布日
3,http_status,HTTP回應碼；200不代表取得貼文
4,return_code,curl退出碼
5,error,curl錯誤文字；成功時空白
6,interpretation,僅回應快照，不是經驗證貼文
7,posts_collected,未知留空，不以0代表無發文
8,complete_window,False，尚未蒐集完整窗口
9,path,回應快照專案相對路徑


### Sample


In [4]:
for name,df in data.items():
    print(name);display(df.head(10))

council_candidates_2026_registration


,candidate_key,election_year,office_type,county_city,electoral_district,registration_date_roc,candidate_name,candidate_name_original,party_recommendation_original,party_recommendation_joined,source_note,source_file,source_page,source_url,qualification_status,captured_date
0,2026|直轄市議員|臺北市第1選舉區|謝閔弘,2026,直轄市議員,臺北市,臺北市第1選舉區,115/08/31,謝閔弘,謝閔弘,時代力量,時代力量,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
1,2026|直轄市議員|臺北市第1選舉區|李文,2026,直轄市議員,臺北市,臺北市第1選舉區,115/08/31,李文,李文,無,無,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
2,2026|直轄市議員|臺北市第1選舉區|陳彥廷,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,陳彥廷,陳彥廷,無,無,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
3,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,鍾佩玲,鍾佩玲,民主進步黨,民主進步黨,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
4,2026|直轄市議員|臺北市第1選舉區|紀建漢,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,紀建漢,紀建漢,無,無,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
5,2026|直轄市議員|臺北市第1選舉區|陳賢蔚,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,陳賢蔚,陳賢蔚,民主進步黨,民主進步黨,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
6,2026|直轄市議員|臺北市第1選舉區|林世宗,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,林世宗,林世宗,民主進步黨,民主進步黨,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
7,2026|直轄市議員|臺北市第1選舉區|黃瀞瑩,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,黃瀞瑩,黃瀞瑩,台灣民眾黨,台灣民眾黨,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
8,2026|直轄市議員|臺北市第1選舉區|林揚庭,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,林揚庭,林揚庭,中國國民黨,中國國民黨,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18
9,2026|直轄市議員|臺北市第1選舉區|陳慈慧,2026,直轄市議員,臺北市,臺北市第1選舉區,115/09/02,陳慈慧,陳慈慧,民主進步黨,民主進步黨,,data/raw/online_20261018/cec_council_metropoli...,1,https://web.cec.gov.tw/api/file/ccd7e51a-5fd0-...,registration_only_not_final_qualification,2026-10-18


current_councilors_20261018


,official_key,official_name,county_city,organization_and_position,party_original,office_type,profile_url,source_file,captured_date,status,term_start,term_end,term_note
0,ER11112AA00029,戴錫欽,臺北市,臺北市議會 議長,中國國民黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
1,ER11112BA00004,張清照,臺中市,臺中市議會 議長,中國國民黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
2,ER11112DA00038,邱莉莉,臺南市,臺南市議會 議長,民主進步黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
3,ER11112EA00010,康裕成,高雄市,高雄市議會 議長,民主進步黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
4,ER11112FA00008,蔣根煌,新北市,新北市議會 議長,中國國民黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
5,ER11112HA00021,邱奕勝,桃園市,桃園市議會 議長,中國國民黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
6,ER11112AA00038,葉林傳,臺北市,臺北市議會 副議長,中國國民黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
7,ER11112BA00002,顏莉敏,臺中市,臺中市議會 副議長,中國國民黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
8,ER11112DA00017,林志展,臺南市,臺南市議會 副議長,民主進步黨,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任
9,ER11112EA00023,曾俊傑,高雄市,高雄市議會 副議長,無,直轄市議員,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/online_20261018/moi_metropolitan.html,2026-10-18,listed_in_moi_current_directory,,,名錄未列任期；不以2022當選自動推定持續在任


legislators_term11_directory_20261018


,legislator_key,name,term,membership_status,party_logo_url,profile_url,source_file,captured_date
0,46752,丁學忠,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
1,46757,牛煦庭,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
2,46758,王世堅,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
3,46759,王育敏,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
4,46760,王定宇,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
5,46761,王美惠,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
6,46762,王鴻薇,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
7,46763,伍麗華Saidhai‧Tahovecahe,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
8,46764,江啟臣,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18
9,46765,何欣純,11,current,https://www.ly.gov.tw/File/Images/Party/PartyL...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=4...,data/raw/online_20261018/ly_members.html,2026-10-18


meta_candidate_account_worklist_20261018


,candidate_key,candidate_name,platform,official_urls,discovered_on,account_status,posts_status,window_count,followers_start,followers_end,updated_date
0,2026-C1-001,郭璽,Facebook,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
1,2026-C1-001,郭璽,Instagram,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
2,2026-C1-001,郭璽,Threads,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
3,2026-C1-002,沈伯洋,Facebook,https://facebook.com/1087764201093899,https://www.pumashen.org/,official_link_previously_saved,not_collected_not_zero,,,,2026-10-18
4,2026-C1-002,沈伯洋,Instagram,https://www.instagram.com/pumashen/,https://www.pumashen.org/,official_link_previously_saved,not_collected_not_zero,,,,2026-10-18
5,2026-C1-002,沈伯洋,Threads,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
6,2026-C1-003,蕭文乾,Facebook,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
7,2026-C1-003,蕭文乾,Instagram,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
8,2026-C1-003,蕭文乾,Threads,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18
9,2026-C1-004,唐新民,Facebook,,,not_verified_not_absent,not_collected_not_zero,,,,2026-10-18


meta_access_attempts_20261018


,subject,url,captured_date,http_status,return_code,error,interpretation,posts_collected,complete_window,path,sha256,response_title,visible_text_chars,content_status
0,Meta Content Library,https://transparency.meta.com/researchtools/me...,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_01.html,da0d3c1a3eaf9fbd40707aeef61e4eee804017beff4006...,Meta Content Library and API | Transparency Ce...,50,page_shell_without_verified_posts
1,Meta research announcement,https://about.fb.com/news/2023/11/new-tools-to...,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_02.html,19e33623156a1cf3d10ba4ce835413c80c0a201a98b311...,New Tools to Support Independent Research,14483,public_documentation
2,SOMAR,https://www.icpsr.umich.edu/sites/somar/meta-c...,2026-10-18,403,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_03.html,fc0b32adb923ca383f12d936331df5a6e259df1fa990b5...,Just a moment...,58,access_challenge
3,Ad Library API,https://www.facebook.com/ads/library/api/,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_04.html,6d6f3b5e5971aa42770ad0910ba16a9fd30fcca825c65e...,廣告檔案庫 API,13397,public_documentation
4,沈伯洋,https://facebook.com/1087764201093899,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_05.html,e6654daaf8515a41a17c6d96444dd436f366b35f7cf502...,台北 沈伯洋,6,page_shell_without_verified_posts
5,林國漳,https://www.facebook.com/linkuochang.yilan,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_06.html,500e367cd7c691907071116246e71842f8d9628c8f506d...,林國漳,3,page_shell_without_verified_posts
6,童子瑋,https://www.facebook.com/wayne201817,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_07.html,166db55b42133547f09b02bb17f1e4832dbcb1e8555496...,童子瑋,3,page_shell_without_verified_posts
7,何欣純,https://www.facebook.com/94achun,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_08.html,4f607fed0095311033066f512f02b9d16fd5de4b660810...,何欣純,3,page_shell_without_verified_posts
8,蘇巧慧,https://www.facebook.com/chiaohui.su,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_09.html,bbf55ba001f1f6d36df884aa37287dcd207db9a1dced3f...,蘇巧慧,3,page_shell_without_verified_posts
9,江啟臣,https://www.facebook.com/johnnyccchiang/,2026-10-18,200,0,,response_only_not_verified_posts,,False,data/raw/online_20261018/meta_access_10.html,86ec64ff936e0df7f72000abf0d05b08de7febac847108...,江啟臣,3,page_shell_without_verified_posts


### Summary Stats


In [5]:
for name,df in data.items():
    print(name)
    display(pd.DataFrame({'variable':df.columns,'missing_N':[(df[c]=='').sum() for c in df.columns],'N':[len(df)]*len(df.columns)}))
    for col in ['office_type','county_city','membership_status','platform','account_status','http_status','content_status']:
        if col in df:
            stats=df[col].value_counts(dropna=False).rename_axis(col).to_frame('N')
            stats['proportion']=stats.N/len(df)
            display(stats)
print('百分比分母為各檔全部列；任期與社群計量欄全缺失，不計算平均數。名錄N不是最終研究樣本數。')


council_candidates_2026_registration


,variable,missing_N,N
0,candidate_key,0,1502
1,election_year,0,1502
2,office_type,0,1502
3,county_city,0,1502
4,electoral_district,0,1502
5,registration_date_roc,0,1502
6,candidate_name,0,1502
7,candidate_name_original,0,1502
8,party_recommendation_original,0,1502
9,party_recommendation_joined,0,1502


,N,proportion
office_type,,
縣市議員,892,0.593875
直轄市議員,610,0.406125


,N,proportion
county_city,,
新北市,112,0.074567
桃園市,109,0.072570
臺中市,102,0.067909
高雄市,101,0.067244
臺北市,98,0.065246
臺南市,88,0.058589
屏東縣,87,0.057923
彰化縣,86,0.057257
雲林縣,73,0.048602


current_councilors_20261018


...

meta_access_attempts_20261018


,variable,missing_N,N
0,subject,0,13
1,url,0,13
2,captured_date,0,13
3,http_status,0,13
4,return_code,0,13
5,error,13,13
6,interpretation,0,13
7,posts_collected,13,13
8,complete_window,0,13
9,path,0,13


,N,proportion
http_status,,
200,12,0.923077
403,1,0.076923


,N,proportion
content_status,,
page_shell_without_verified_posts,10,0.769231
public_documentation,2,0.153846
access_challenge,1,0.076923


百分比分母為各檔全部列；任期與社群計量欄全缺失，不計算平均數。名錄N不是最終研究樣本數。


### Assertions


In [6]:
c=data['council_candidates_2026_registration'];m=data['current_councilors_20261018'];l=data['legislators_term11_directory_20261018'];w=data['meta_candidate_account_worklist_20261018'];a=data['meta_access_attempts_20261018']
assert len(c)==1502 and c.candidate_key.is_unique and c.county_city.nunique()==22
assert c.office_type.value_counts().to_dict()=={'縣市議員':892,'直轄市議員':610}
assert len(m)==872 and m.official_key.is_unique and m.county_city.nunique()==22
assert l.legislator_key.is_unique and l.membership_status.value_counts().to_dict()=={'current':113,'former':10}
assert len(w)==243 and not w.duplicated(['candidate_key','platform']).any()
assert (w.window_count=='').all() and (w.followers_start=='').all() and (w.followers_end=='').all()
assert (a.complete_window=='False').all() and (a.posts_collected=='').all()
print('Checks passed. No final sample, lifetime identity joins, post counts or causal findings are certified.')


Checks passed. No final sample, lifetime identity joins, post counts or causal findings are certified.


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。